# GF($2^m$) Multiplication

Adapted from frozen Qualtran v0.7.0 at bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2024 Google LLC, Apache-2.0. All seven original sections, nine instances, parameters, register topics, signatures and one-level graphs are retained. Native names and tables replace the source presentation. The plus-equal prose uses the actual scalar map z + xy, correcting the original text typo. Field words are big-endian. Polynomial array examples display little-endian source coefficient lanes.

Compiler owns gates, output mapping and caller-owned scratch. Simulation owns field values. Resource graphs describe source declarations and emit zero gates. The executable Karatsuba owner uses a true linear inverse. The frozen source emits its forward linear network for that inverse, with a preserved GF(4) counterexample. Full-source parity for that path remains open.


In [ ]:
from IPython.display import SVG, display
from examples.algorithms.field_multiplication_catalog import field_multiplication_examples, field_multiplication_example_graph
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
examples = field_multiplication_examples()


## `Field product`
Out of place multiplication over GF($2^m$).

The bloq implements out of place multiplication of two quantum registers storing elements
from GF($2^m$) using construction described in Ref[1], which extends the classical construction
of Ref[2].

To multiply two m-bit inputs $a = [a_0, a_1, ..., a_{m-1}]$ and $b = [b_0, b_1, ..., b_{m-1}]$,
the construction computes the output vector $c$ via the following three steps:
    1. Compute $e = U.b$ where $U$ is an upper triangular matrix constructed using $a$.
    2. Compute $Q.e$ where $Q$ is an $m \times (m - 1)$ reduction matrix that depends upon the
        irreducible polynomial $P(x)$ of the galois field $GF(2^m)$. The i'th column of the
        matrix corresponds to coefficients of the polynomial $x^{m + i} % P(x)$. This matrix $Q$
        is a linear reversible circuit that can be implemented only using CNOT gates.
    3. Compute $d = L.b$ where $L$ is a lower triangular matrix constructed using $a$.
    4. Compute $c = d + Q.e$ to obtain the final product.

Steps 1 and 3 are performed using $n^2$ Toffoli gates and step 2 is performed only using CNOT
gates.

#### Parameters
 - `bitsize`: The degree $m$ of the galois field $GF(2^m)$. Also corresponds to the number of qubits in each of the two input registers $a$ and $b$ that should be multiplied.
 - `plus_equal_prod`: If True, implements the `PlusEqualProduct` version that applies the map $|x\rangle |y\rangle |z\rangle \rightarrow |x\rangle |y\rangle |z + xy\rangle$. 

#### Registers
 - `x`: Input THRU register of size $m$ that stores elements from $GF(2^m)$.
 - `y`: Input THRU register of size $m$ that stores elements from $GF(2^m)$.
 - `result`: Register of size $m$ that stores the product $x * y$ in $GF(2^m)$. If plus_equal_prod is True - result is a THRU register and stores $result + x * y$. If plus_equal_prod is False - result is a RIGHT register and stores $x * y$.  

#### References
 - [On the Design and Optimization of a Quantum Polynomial-Time Attack on Elliptic Curve Cryptography](https://arxiv.org/abs/0710.1093). 
 - [Low complexity bit parallel architectures for polynomial basis multiplication over GF(2m)](https://ieeexplore.ieee.org/abstract/document/1306989).


In [ ]:
examples[0:2]


### Example Instances


In [ ]:
field_product_accumulate = examples[0]
field_product_accumulate


In [ ]:
field_product_symbolic = examples[1]
field_product_symbolic


#### Graphical Signature


In [ ]:
display(SVG(field_signature_svg(examples[:2])))


### Call Graph


In [ ]:
graph = field_multiplication_example_graph(field_product_accumulate)
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
graph.leaf_counts


## `Constant field scale`
Multiply by constant $f(x)$ modulo $m(x)$. Both $f(x)$ and $m(x)$ are constants.

#### Parameters
 - `const`: The multiplication constant which is an element of the given field.
 - `galois_field`: The galois field that defines the arithmetics. 

#### Registers
 - `g`: The polynomial coefficients (in GF(2)). 

#### References
 - [Space-efficient quantum multiplication of polynomials for binary finite fields with     sub-quadratic Toffoli gate count](https://arxiv.org/abs/1910.02849v2). Algorithm 1


In [ ]:
examples[2:4]


### Example Instances


In [ ]:
field_scale_value = examples[2]
field_scale_value


In [ ]:
field_scale_polynomial = examples[3]
field_scale_polynomial


#### Graphical Signature


In [ ]:
display(SVG(field_signature_svg(examples[2:4])))


### Call Graph


In [ ]:
graph = field_multiplication_example_graph(field_scale_value)
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
graph.leaf_counts


## `Scaled polynomial product`
Out of place multiplication of $(1 + x^k) fg$

Applies the transformation
$$
\ket{f}\ket{g}\ket{h} \rightarrow \ket{f}{\ket{g}}\ket{h \oplus (1+x^k)fg}
$$

Note: While this construction follows Algorithm2 of https://arxiv.org/abs/1910.02849v2,
it has a slight modification. Namely that the original construction doesn't work in
some cases where $k < n$. However reversing the order of the first set of CNOTs (line 2)
makes the construction work for all $k \leq n+1$.

#### Parameters
 - `n`: The degree of the polynomial ($2^n$ is the size of the galois field).
 - `k`: An integer specifing the shift $1 + x^k$ (or $1 + 2^k$ for galois fields.) 

#### Registers
 - `f`: The first polynomial.
 - `g`: The second polyonmial.
 - `h`: The target polynomial. 

#### References
 - [Space-efficient quantum multiplication of polynomials for binary finite fields with     sub-quadratic Toffoli gate count](https://arxiv.org/abs/1910.02849v2). Algorithm 2


In [ ]:
examples[4]


### Example Instances


In [ ]:
polynomial_scaled_product = examples[4]
polynomial_scaled_product


#### Graphical Signature


In [ ]:
display(SVG(field_signature_svg([polynomial_scaled_product])))


### Call Graph


In [ ]:
graph = field_multiplication_example_graph(polynomial_scaled_product)
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
graph.leaf_counts


## `Binary polynomial product`
Out of place multiplication of binary polynomial multiplication.

Applies the transformation
$$
\ket{f}\ket{g}\ket{h} \rightarrow \ket{f}{\ket{g}}\ket{h \oplus fg}
$$

The toffoli cost of this construction is $n^{\log_2{3}}$, while CNOT count is
upper bounded by $(10 + \frac{1}{3}) n^{\log_2{3}}$.

#### Parameters
 - `n`: The degree of the polynomial ($2^n$ is the size of the galois field). 

#### Registers
 - `f`: The first polynomial.
 - `g`: The second polyonmial.
 - `h`: The target polynomial. 

#### References
 - [Space-efficient quantum multiplication of polynomials for binary finite fields with     sub-quadratic Toffoli gate count](https://arxiv.org/abs/1910.02849v2). Algorithm 3


In [ ]:
examples[5]


### Example Instances


In [ ]:
polynomial_product = examples[5]
polynomial_product


#### Graphical Signature


In [ ]:
display(SVG(field_signature_svg([polynomial_product])))


### Call Graph


In [ ]:
graph = field_multiplication_example_graph(polynomial_product)
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
graph.leaf_counts


## `Forward monomial shift`
Multiplies by $2^k$ (or $x^k$ for polynomials) modulo the given irreducible polynomial.

Applies the transformation
$$
    \ket{f} \rightarrow \ket{x^k f \mod m(x)}
$$

Where the modulus $m(x)$ is the irreducible polynomial defining the galois field arithmetic.

#### Parameters
 - `m_x`: The irreducible polynomial that defines the galois field.
 - `k`: The number of shifts (i.e. the exponent of $2$ or $x$). 

#### Registers
 - `f`: The number (polynomial) to shift. 

#### References
 - [Space-efficient quantum multiplication of polynomials for binary finite fields with     sub-quadratic Toffoli gate count](https://arxiv.org/abs/1910.02849v2). Section 3.1


In [ ]:
examples[6]


### Example Instances


In [ ]:
field_shift_forward = examples[6]
field_shift_forward


#### Graphical Signature


In [ ]:
display(SVG(field_signature_svg([field_shift_forward])))


### Call Graph


In [ ]:
graph = field_multiplication_example_graph(field_shift_forward)
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
graph.leaf_counts


## `Karatsuba field product`
Multiplies two GF($2^n$) numbers (or binary polynomials) modulo $m(x)$.

Applies the transformation
$$
    \ket{f}\ket{g} \rightarrow \ket{f} \ket{g} \ket{f*g \mod m(x)}
$$

Where the modulus $m(x)$ is the irreducible polynomial defining the galois field arithmetic.
The toffoli complexity is $n^{\log_2{3}}$

#### Parameters
 - `m_x`: The irreducible polynomial that defines the galois field. 

#### Registers
 - `x`: A TRHU register representing the first number (or polynomial).
 - `y`: A TRHU register representing the second number (polynomial).
 - `result`: The result (a RIGHT register). 

#### References
 - [Space-efficient quantum multiplication of polynomials for binary finite fields with     sub-quadratic Toffoli gate count](https://arxiv.org/abs/1910.02849v2). Algorithm 4.


In [ ]:
examples[7]


### Example Instances


In [ ]:
field_product_karatsuba = examples[7]
field_product_karatsuba


#### Graphical Signature


In [ ]:
display(SVG(field_signature_svg([field_product_karatsuba])))


### Call Graph


In [ ]:
graph = field_multiplication_example_graph(field_product_karatsuba)
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
graph.leaf_counts


## `Inverse monomial shift`
Divides by $2^k$ (or $x^k$ for polynomials) modulo the given irreducible polynomial.

Applies the transformation
$$
    \ket{f} \rightarrow \ket{x^{-k} f \mod m(x)}
$$

Where the modulus $m(x)$ is the irreducible polynomial defining the galois field arithmetic.

#### Parameters
 - `m_x`: The irreducible polynomial that defines the galois field.
 - `k`: The number of shifts (i.e. the exponent of $2$ or $x$). 

#### Registers
 - `f`: The number (polynomial) to shift.


In [ ]:
examples[8]


### Example Instances


In [ ]:
field_shift_inverse = examples[8]
field_shift_inverse


#### Graphical Signature


In [ ]:
display(SVG(field_signature_svg([field_shift_inverse])))


### Call Graph


In [ ]:
graph = field_multiplication_example_graph(field_shift_inverse)
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
graph.leaf_counts


## Execute products and measured erasure through append

The existing native field example computes a GF(8) product, clean inverse, constant scale, monomial round trip, arbitrary-target polynomial product and product erasure through circuit.append(op). It verifies all complex amplitudes for each of 16 sampled erasure trajectories. Exhaustive small measurement-branch tests are maintained independently. Source category models omit erasure measurements and parity, so actual primitive counts are printed separately. Output binding SWAPs are listed separately from free logical relabelings. Work ownership and clean-input preconditions are explicit in each Compiler operation.


In [ ]:
from examples.algorithms.binary_field_arithmetic import main
from flagquantum.compiler.registers.finite_field.multiplication_model import field_multiplication_gate_model
main()
print("Source GF(8) quadratic model", field_multiplication_gate_model(3, modulus=11))
print("Source GF(8) Karatsuba model", field_multiplication_gate_model(3, modulus=11, algorithm="karatsuba"))
print("Source GF(8) erasure model", field_multiplication_gate_model(3, algorithm="erasure"))
